# DrugCLIP screening with Fpocket — no known binding site

The hardest case: a predicted structure and **no relative with a bound ligand**
to borrow a pocket from. Here you have to *detect* the site rather than
transplant it.

This is the weakest of the three routes, and the screen-pipeline README says so:

> Tools like Fpocket can be used, but **only half of those pockets are precise
> enough for DrugCLIP**

The paper measures the cost — EF1% drops from 29.3 (pocket aligned from a holo
structure) to **19.0** (Fpocket). Use `run_screen_alphafold.ipynb` instead
whenever any relative with a bound ligand exists.

### The problem this notebook addresses

Fpocket returns dozens of candidate cavities (32 for PGK2), ranked by its own
scoring. If the top one is only about half likely to be the real site, taking it
alone is a coin flip. Taking several improves the odds — but combining them
naively does not work.

The repo's `retrieve_mols()` max-pools raw scores:

```python
res = pocket_reps @ mol_reps.T
res = res.max(axis=0)          # no normalisation
```

The paper is explicit that this is wrong:

> because different pockets may have varying score ranges, this can introduce
> bias when applying max pooling. To address this, we normalize the scores using
> an **adjusted robust z-score** before performing the max pooling

$$z_i = \frac{x_i - \mathrm{median}(X)}{1.48 \cdot \mathrm{MAD}(X)}$$

Without it, whichever cavity happens to produce larger raw similarities
dominates every molecule's score, whether or not it is the real site — a large
greasy cavity would swamp a genuine pocket.

So this notebook scores each pocket **separately**, normalises per pocket, then
max-pools. It reports the per-pocket rankings too, because disagreement between
pockets is itself informative.

## Configuration

In [ ]:
from pathlib import Path
import shutil

# Found by walking up from wherever this runs, so the same notebook works on
# any machine and survives a git pull. unimol/ marks the repo root.
REPO     = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "unimol").is_dir())
PIPELINE = REPO.parent / "DrugCLIP_screen_pipeline"
WORK     = REPO / "data" / "pgk2_fpocket"      # gitignored

UNIPROT  = "P07205"       # human PGK2
N_POCKETS = 5             # how many of Fpocket's ranked pockets to screen
RANK_BY  = "druggability" # "druggability" or "score"

MOL_CSV  = REPO / "data" / "pgk2" / "sample_molecules.csv"
N_CONF   = 10
WEIGHTS  = REPO / "model" / "dude_identity_90.pt"
# Whatever is on PATH, else the usual build location. See SETUP.md to build it.
FPOCKET  = Path(shutil.which("fpocket") or Path.home() / "fpocket" / "bin" / "fpocket")

# Keyed on the checkpoint, not the pocket: molecule embeddings depend only on
# the weights, so all three pocket strategies share one cache. Shared with the
# other single-model notebooks.
EMB_DIR     = REPO / "data" / "emb" / WEIGHTS.stem
WORK.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK)
print("fpocket: ", "found" if FPOCKET.exists() else "MISSING - see SETUP.md")
print("weights: ", "found" if WEIGHTS.exists() else "MISSING")

# --- output identity -------------------------------------------------------
# METHOD names the pocket strategy; results are written as
#   data/results/<molecule-csv-stem>__<METHOD>.csv
# so the same test set can be run through all three notebooks and compared.
METHOD      = "alphafold_fpocket"   # AlphaFold model, pocket detected by Fpocket
RESULTS_DIR = REPO / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / (Path(MOL_CSV).stem + "__" + METHOD + ".csv")
print("results ->", RESULTS_CSV)

# Named after the source CSV so different compound sets never collide, and so
# the embedding cache - which is keyed on this filename - stays correct per
# library. Shared with build_molecules.ipynb.
LMDB_DIR  = REPO / "data" / "lmdb"
LMDB_DIR.mkdir(parents=True, exist_ok=True)
MOLS_LMDB = LMDB_DIR / (Path(MOL_CSV).stem + ".lmdb")

In [ ]:
import sys, json, re, pickle, shutil, subprocess, urllib.request
import numpy as np, pandas as pd, lmdb, torch
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.*")
sys.path.insert(0, str(REPO))

def read_lmdb(path):
    env = lmdb.open(str(path), subdir=False, readonly=True, lock=False,
                    readahead=False, meminit=False, max_readers=256)
    with env.begin() as txn:
        out = [pickle.loads(v) for _, v in txn.cursor()]
    env.close()
    return out

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

## Step 1 — Structure, then Fpocket

The AlphaFold model is a download, not a computation — the database already
holds predictions for essentially every human protein.

In [ ]:
api = "https://alphafold.ebi.ac.uk/api/prediction/" + UNIPROT
meta = json.loads(urllib.request.urlopen(api).read())[0]
af_pdb = WORK / "AF_model.pdb"
if not af_pdb.exists():
    urllib.request.urlretrieve(meta["pdbUrl"], af_pdb)
print("gene", meta.get("gene"), "| version", meta.get("latestVersion"))

b = np.array([float(l[60:66]) for l in open(af_pdb) if l.startswith("ATOM")])
print("plDDT median %.1f | above 90: %.0f%% | below 70: %.1f%%"
      % (np.median(b), 100*(b > 90).mean(), 100*(b < 70).mean()))

In [ ]:
out_dir = WORK / "AF_model_out"
if not out_dir.exists():
    r = subprocess.run([str(FPOCKET), "-f", "AF_model.pdb"],
                       cwd=str(WORK), capture_output=True, text=True)
    print(r.stdout.strip()[-200:] or r.stderr[-400:])

pqrs = sorted((out_dir / "pockets").glob("pocket*_vert.pqr"))
print("Fpocket found", len(pqrs), "candidate cavities")

## Step 2 — Rank and select

Fpocket scores every cavity. **Druggability Score** estimates how likely a
cavity is to bind a drug-like molecule, and is usually the more relevant of the
two for screening; **Pocket Score** is its general-purpose ranking.

Neither is reliable enough to trust blindly — that is the whole reason for
taking several.

In [ ]:
info_txt = (out_dir / "AF_model_info.txt").read_text()
blocks = re.split(r"Pocket (\d+) :", info_txt)[1:]

rows = []
for num, body in zip(blocks[0::2], blocks[1::2]):
    def grab(key):
        m = re.search(re.escape(key) + r"\s*:\s*([-\d.]+)", body)
        return float(m.group(1)) if m else np.nan
    rows.append({"pocket": int(num),
                 "score": grab("Score"),
                 "druggability": grab("Druggability Score"),
                 "volume": grab("Volume"),
                 "alpha_spheres": grab("Number of Alpha Spheres")})

pk = pd.DataFrame(rows).sort_values(RANK_BY, ascending=False).reset_index(drop=True)
selected = pk.head(N_POCKETS)
display(pk.head(10).style.set_caption("Fpocket cavities, ranked by " + RANK_BY))
print("selected pockets:", list(selected["pocket"]))

## Step 3 — Turn each cavity into a pocket

Fpocket outputs *alpha-sphere centres* filling a cavity, not a ligand. The
pipeline's trick is to write those centres out as a fake ligand residue named
`STP`, so the ordinary 6 A rule has something to measure from:

```python
lig_res = Residue.Residue(('H_STP', 1, ' '), 'STP', ' ')   # build_fpocket_complex.py
```

Nothing downstream can tell the difference between this and a real bound ligand.

In [ ]:
sys.path.insert(0, str(PIPELINE))
from build_fpocket_complex import pqr_parser, build_complex

pdb_dir = WORK / "pdb_in"
if pdb_dir.exists():
    shutil.rmtree(pdb_dir)
pdb_dir.mkdir()

made = []
for p in selected["pocket"]:
    pqr = out_dir / "pockets" / ("pocket%d_vert.pqr" % p)
    stub = pdb_dir / ("p%02d.pdb" % p)
    shutil.copy(af_pdb, stub)
    build_complex(str(stub), pqr_parser(str(pqr)))   # writes p##_STP.pdb
    stub.unlink()                                     # keep only the _STP file
    made.append(pdb_dir / ("p%02d_STP.pdb" % p))
    print("pocket %2d -> %s  (%d alpha spheres)"
          % (p, made[-1].name, len(pqr_parser(str(pqr)))))

In [ ]:
# one LMDB per pocket, so each can be scored on its own
pocket_lmdbs = {}
for p, f in zip(selected["pocket"], made):
    one = WORK / ("pocket_%02d_dir" % p)
    if one.exists():
        shutil.rmtree(one)
    one.mkdir()
    shutil.copy(f, one / f.name)
    r = subprocess.run([sys.executable, str(PIPELINE / "pocket_from_pdb.py"),
                        str(one), "--name", "pocket"],
                       capture_output=True, text=True, cwd=str(PIPELINE))
    lm = one / "pocket.lmdb"
    if not lm.exists():
        print("pocket %2d FAILED - pocket_from_pdb.py swallows errors silently" % p)
        continue
    recs = read_lmdb(lm)
    if not recs:
        print("pocket %2d produced an EMPTY lmdb" % p)
        continue
    pocket_lmdbs[p] = lm
    print("pocket %2d -> %3d atoms  elements=%s"
          % (p, len(recs[0]["pocket_atoms"]), sorted(set(recs[0]["pocket_atoms"]))))

assert pocket_lmdbs, "no usable pockets"

## Step 4 — Molecule library

Unchanged from the other notebooks.

In [ ]:

# Reuse a library built by build_molecules.ipynb, which generates conformers in
# parallel. Conformer generation is the slowest step and does not depend on the
# pocket, the weights or the method, so it is worth doing once.
if MOLS_LMDB.exists():
    _recs = read_lmdb(MOLS_LMDB)
    _meta = next((r for r in _recs if isinstance(r, dict) and "source_csv" in r), None)
    records = [r for r in _recs if "smi" in r]
    print(f"reusing {MOLS_LMDB.name}: {len(records)} molecules"
          + (f" (from {_meta['source_csv']})" if _meta else ""))
else:
    print(f"{MOLS_LMDB.name} not found - building it here.")
    print("For a large library, run build_molecules.ipynb instead; it uses all cores.")

    df = pd.read_csv(MOL_CSV)

    def to_record(name, smi, n_conf=N_CONF):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None, "unparseable SMILES"
        mol = Chem.AddHs(mol)
        if len(AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, randomSeed=42,
                                          pruneRmsThresh=1)) == 0:
            return None, "no conformer embedded"
        try:
            AllChem.MMFFOptimizeMoleculeConfs(mol)
        except Exception:
            pass
        mol = Chem.RemoveHs(mol)
        return {"atoms": [a.GetSymbol() for a in mol.GetAtoms()],
                "coordinates": [np.array(mol.GetConformer(c).GetPositions(),
                                         dtype=np.float32)
                                for c in range(mol.GetNumConformers())],
                "smi": smi, "name": name}, None

    records, failures = [], []
    for _, row in df.iterrows():
        rec, err = to_record(row["name"], row["smiles"])
        (records.append(rec) if rec else failures.append((row["name"], err)))
    print("built", len(records), "| failed", len(failures))

    vocab = set((REPO / "data" / "dict_mol.txt").read_text().split())
    unknown = set(a for r in records for a in r["atoms"]) - vocab
    print("elements not in dict_mol.txt:", sorted(unknown) or "none")

    if MOLS_LMDB.exists():
        MOLS_LMDB.unlink()
    env = lmdb.open(str(MOLS_LMDB), subdir=False, readonly=False, lock=False,
                    readahead=False, meminit=False, map_size=1099511627776)
    with env.begin(write=True) as txn:
        for i, r in enumerate(records):
            txn.put(str(i).encode("ascii"), pickle.dumps(r))
    env.close()
    print(MOLS_LMDB.name, ":", len(read_lmdb(MOLS_LMDB)), "records")

## Step 5 — Score each pocket separately

One screening run per pocket. Molecule embeddings are cached after the first
run and reused, so the extra cost is only encoding each pocket — cheap.

In [ ]:
for f in EMB_DIR.glob("*.pkl"):
    f.unlink()      # cache is keyed on filename, not contents

def screen(pocket_lmdb):
    out = EMB_DIR / "ranked_compounds.txt"
    if out.exists():
        out.unlink()
    cmd = [sys.executable, "./unimol/retrieval.py",
           "--user-dir", "./unimol", str(REPO / "data"),
           "--valid-subset", "test", "--results-path", str(WORK / "results"),
           "--num-workers", "0", "--ddp-backend", "c10d", "--batch-size", "8",
           "--task", "drugclip", "--loss", "in_batch_softmax", "--arch", "drugclip",
           "--max-pocket-atoms", "256", "--seed", "1",
           "--path", str(WEIGHTS), "--log-interval", "100", "--log-format", "simple",
           "--mol-path", str(MOLS_LMDB), "--pocket-path", str(pocket_lmdb),
           "--emb-dir", str(EMB_DIR)]
    if not torch.cuda.is_available():
        cmd.append("--cpu")
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO))
    if not out.exists():
        raise RuntimeError("screening failed:\n" + r.stderr[-2000:])
    d = {}
    for line in out.read_text().splitlines():
        if line.strip():
            smi, s = line.rsplit("\t", 1)
            d[smi] = float(s)
    return d

per_pocket = {}
for p, lm in pocket_lmdbs.items():
    per_pocket[p] = screen(lm)
    print("pocket %2d scored" % p)

## Step 6 — Normalise, then combine

The adjusted robust z-score, exactly as the paper defines it:

$$z_i = \frac{x_i - \mathrm{median}(X)}{1.48 \cdot \mathrm{MAD}(X)}$$

Median and MAD are used instead of mean and standard deviation because they are
insensitive to the handful of high scorers — which are precisely the molecules
of interest, and which would otherwise drag the normalisation around.

The 1.48 factor makes MAD comparable to a standard deviation for normally
distributed data, so the resulting z-scores read on a familiar scale.

### A caveat on reading any of this

These scores are cosine similarities turned into z-scores *within this molecule
set*. With a small library the median and MAD are themselves unstable, so a
z-score here is not comparable to the z > 3 threshold in the screen-pipeline
README — that assumes a background of thousands of molecules.

For a real campaign: spike the compounds of interest into 10k-100k background
molecules, and read the z-score against that distribution.

And the underlying caveat remains — Fpocket's pockets are right about half the
time. A high score against a cavity that is not the real binding site means
nothing. If any relative with a bound ligand exists, use
`run_screen_alphafold.ipynb` instead.

## Results

Written to `RESULTS_CSV` with a schema shared by all three notebooks, so runs
can be loaded together and compared.

| column | meaning |
| --- | --- |
| `rank` | 1 = best |
| `score_raw` | raw cosine similarity (max over pockets, where there are several) |
| `zscore` | robust z across this molecule set: (x - median) / (1.48 * MAD) |
| `method` | which pocket strategy produced it |

**Compare by rank, not by score.** Cosine values are not calibrated across
different pockets - 0.14 against one pocket does not mean what 0.14 means
against another. Within a single run the ordering is meaningful; across runs
only the ordering is.

For a single pocket, `zscore` is a monotone transform of `score_raw`, so the two
give the same ranking. Where several pockets are combined they differ, because
the z-score is computed per pocket before pooling.

And the standing caveat: a z-score over a few hundred molecules is not the z > 3
of the screen-pipeline README, which assumes a background of thousands.

In [ ]:
def robust_z(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return np.zeros_like(x) if mad == 0 else (x - med) / (1.48 * mad)

smis  = [r["smi"] for r in records]
names = dict((r["smi"], r["name"]) for r in records)

# raw[pocket] = cosine similarity of every molecule against that pocket
raw = pd.DataFrame({p: [per_pocket[p][s] for s in smis] for p in per_pocket},
                   index=smis)
# normalise each pocket before pooling, as the paper requires
z = raw.apply(robust_z, axis=0)

EXTRA_COLS = ["best_pocket"]
res = pd.DataFrame({
    "name": [names[s] for s in smis],
    "smiles": smis,
    "score_raw": raw.max(axis=1).values,   # what plain max-pooling would give
    "zscore": z.max(axis=1).values,        # normalised first - the correct form
    "best_pocket": z.idxmax(axis=1).values,
})
res["score_sort"] = res["zscore"]

res = res.sort_values("score_sort", ascending=False).reset_index(drop=True)
res["rank"] = np.arange(1, len(res) + 1)
res["method"] = METHOD
res = res[["rank", "name", "smiles", "score_raw", "zscore", "method"] + EXTRA_COLS]
res.to_csv(RESULTS_CSV, index=False)

print("wrote", RESULTS_CSV, "(%d molecules)" % len(res))
display(res.head(20))

In [ ]:
# do the pockets agree with each other?
zz = z.copy()
zz.index = [names[s] for s in smis]
zz.columns = ["pocket %d" % c for c in zz.columns]
display(zz.round(2))

corr = zz.corr(method="spearman")
import numpy as _np
print("mean pairwise Spearman between pockets: %.2f"
      % corr.values[_np.triu_indices_from(corr, k=1)].mean())
print()
print("Agreement suggests the pockets describe overlapping sites. Disagreement")
print("means they describe genuinely different cavities - which is why several")
print("are screened when the true site is unknown.")